<h2>Agent 1 — Evaluator</h2>

In [2]:
def evaluator_agent(cv_text, job_text):
    prompt = f"""
You are an ATS evaluator.

Return JSON:
{{
  "score": 0-100,
  "keyword_match": 0-100,
  "experience_match": 0-100,
  "skills_missing": [],
  "critical_missing": [],
  "summary": ""
}}

Job:
{job_text}

CV:
{cv_text}
"""
    return call_ai(prompt)

<h2>Agent 2 — Gatekeeper</h2>
Decide user should apply this job or not

In [3]:
def gatekeeper_agent(eval_result):
    prompt = f"""
Decide whether to optimize CV.

Rules:
- If critical skills missing → REJECT
- If experience strong but keyword weak → OPTIMIZE
- If too far from job → REJECT

Return:
{{
  "decision": "REJECT | OPTIMIZE",
  "reason": "",
  "confidence": 0-100
}}

Evaluation:
{eval_result}
"""
    return call_ai(prompt)

<h2>Agent 3 — Optimizer</h2>

In [4]:
def optimizer_agent(cv_text, job_text):
    prompt = f"""
Rewrite CV to improve ATS score.

Return structured JSON:
{{
  "summary": "",
  "skills": [],
  "experience": []
}}

Rules:
- Add missing skills if reasonable
- Do NOT fabricate experience
- Improve bullet points with metrics

Job:
{job_text}

CV:
{cv_text}
"""
    return call_ai(prompt)

<h2>Agent 4 — Validator</h2>
Detect fake improvements

In [5]:
def validator_agent(old_eval, new_eval):
    prompt = f"""
Check if CV improvement is REAL.

Rules:
- If score improves but skills still missing → FAKE
- If improvement < 10 → NOT USEFUL

Return:
{{
  "is_valid": true/false,
  "improvement_score": number,
  "issues": []
}}

Before:
{old_eval}

After:
{new_eval}
"""
    return call_ai(prompt)

<h2>Agent 5 — Final Decision</h2>
Give results for user to decide applying or not

In [6]:
def decision_agent(old_eval, new_eval, validation):
    prompt = f"""
Make final decision:

Options:
- REJECT
- APPLY_WITH_CAUTION
- APPLY

Return:
{{
  "final_decision": "",
  "confidence": 0-100,
  "reason": "",
  "should_apply": true/false
}}

Before:
{old_eval}

After:
{new_eval}

Validation:
{validation}
"""
    return call_ai(prompt)

<h2>Agent 6 — Formatter</h2>
Generate new CV

<h2>Orchestrator (Main Brain)</h2>

In [7]:
def run_pipeline(cv_text, job_text):
    # 1. Evaluate
    eval1 = evaluator_agent(cv_text, job_text)

    # 2. Gatekeeper
    gate = gatekeeper_agent(eval1)

    if gate["decision"] == "REJECT":
        return {
            "status": "REJECTED",
            "reason": gate
        }

    # 3. Optimize
    new_cv = optimizer_agent(cv_text, job_text)

    # 4. Convert to text
    new_text = str(new_cv)

    # 5. Re-evaluate
    eval2 = evaluator_agent(new_text, job_text)

    # 6. Validate
    validation = validator_agent(eval1, eval2)

    if not validation["is_valid"]:
        return {
            "status": "REJECTED_AFTER_OPTIMIZATION",
            "validation": validation
        }

    # 7. Final decision
    final = decision_agent(eval1, eval2, validation)

    return {
        "status": "DONE",
        "before": eval1,
        "after": eval2,
        "final": final,
        "cv": new_cv
    }

#test
# run_pipeline()